# XGBoost Survival Embeddings (XGBSE) – Training Pipeline

Questa prima parte del notebook descrive il processo di addestramento per il modello di analisi di sopravvivenza basato su gradient boosting.
Le scelte principali riguardano la **gestione delle feature**, la **strategia di campionamento degli iperparametri** e le **modalità di salvataggio dell’artefatto**, pensate per la riproducibilità e la completezza dell’analisi explainable.

---

### Fase 1: Caricamento e Preprocessing – Nessuna Feature Selection

Il dataset viene caricato (percorso relativo `../data/dataset.csv`) e i placeholder `-1` vengono convertiti in `NaN` per sfruttare la gestione nativa dei valori mancanti da parte di XGBoost.

**Non viene applicata alcuna selezione manuale delle feature.** La scelta è motivata dalla capacità intrinseca di XGBoost di:

- apprendere automaticamente la direzione ottimale nei nodi decisionali in presenza di `NaN`;
- gestire efficacemente dataset con molte feature, anche ridondanti, grazie ai meccanismi di regolarizzazione;
- evitare la perdita di informazioni potenzialmente utili che potrebbero derivare da una selezione a priori.

Vengono quindi mantenute **tutte le variabili numeriche** presenti nel dataset, ad esclusione degli identificativi (`cases.case_id`) e delle colonne target (tempo ed evento): 33 feature in totale.

---

### Fase 2: Suddivisione Train / Test

Il dataset viene suddiviso in training (75%) e test (25%), con **stratificazione sull’esito** (`vital_status_DEAD`) per garantire che la proporzione di eventi censurati e osservati sia bilanciata nei due insiemi.
Il seed `RANDOM_STATE = 25` assicura la riproducibilità dello split.

---

### Fase 3: Configurazione degli Iperparametri – Campionamento Deterministico

La strategia di ricerca degli iperparametri è pensata per garantire una **copertura controllata e riproducibile** dello spazio delle combinazioni.
Invece di definire distribuzioni di probabilità (es. uniformi), viene generata in modo deterministico una lista di **150 insiemi di parametri** utilizzando un generatore casuale con seed fisso. Questa lista viene poi passata a `RandomizedSearchCV` come dizionario con chiave `"xgb_params"`. Con `n_iter=150` la ricerca esplora **tutte** le combinazioni pre‑generate, assicurando che non vi sia alcuna aleatorietà residua nel campionamento.

I valori esplorati riflettono una filosofia di **forte regolarizzazione**, adatta a dataset di dimensioni contenute (Head & Neck Cancer):

- `n_estimators`: 200, 300, 400, 500 → compromesso tra capacità e tempo di addestramento.
- `learning_rate`: 0.005, 0.01, 0.02, 0.03 → valori volutamente bassi per ridurre il contributo di ogni albero e contrastare l’overfitting.
- `max_depth`: 2, 3, 4 → alberi poco profondi per mantenere bassa la varianza.
- `min_child_weight`: 5, 10, 15, 20, 30 → richiede un peso minimo per ogni nodo, scoraggiando splits su campioni troppo piccoli.
- `gamma`: 0.5, 1, 2, 5 → riduzione minima del loss per giustificare una suddivisione.
- `subsample`: 0.5, 0.6, 0.7, 0.8 → campionamento delle righe per ogni albero.
- `colsample_bynode`: 0.3, 0.5, 0.7, 1.0 → frazione di feature considerate per ogni nodo.
- `reg_alpha`: 0.5, 1, 5, 10, 20 → regolarizzazione L1.
- `reg_lambda`: 1, 5, 10, 20, 50 → regolarizzazione L2.

Viene adottata l’impostazione di `tree_method = "exact"` (anziché l’istogramma approssimato), resa possibile dalla dimensione contenuta del dataset e necessaria per una matematica di split più precisa.

---

### Fase 4: Ottimizzazione con XGBSEDebiasedBCE

Il modello prescelto è **XGBSEDebiasedBCE** della libreria `xgbse`. Questo estimatore lavora in due stadi: un modello XGBoost di sopravvivenza viene addestrato sui dati e usato come estrattore di feature (per ogni paziente, la foglia raggiunta in ciascun albero: il *leaf embedding*); su questo embedding viene poi addestrata una serie di regressioni logistiche, una per finestra temporale, ognuna delle quali stima la probabilità di sopravvivenza a quell’istante. Si ottiene così direttamente una stima della funzione di sopravvivenza. Nella versione *debiased* ogni paziente censurato viene escluso dalle finestre successive alla sua censura, invece di essere trattato come sopravvissuto.

All’interno dei parametri dell’XGBoost sottostante viene specificato:

- `objective = "survival:cox"` – il modello XGBoost che genera l’embedding è pensato per ottimizzare la verosimiglianza parziale di Cox (l’alternativa prevista dalla libreria è `survival:aft`).
- `eval_metric = "cox-nloglik"` – metrica di valutazione interna.

Per la ricerca degli iperparametri, viene definita una **funzione di scoring personalizzata** `score_xgbse` che:

1. predice le curve di sopravvivenza con `estimator.predict(X_val)`;
2. trasforma le curve in un **punteggio di rischio** tramite `risk = 1.0 - preds_df.values.mean(axis=1)` (media delle probabilità di sopravvivenza nel tempo, complemento a 1);
3. calcola il **concordance index** tra i rischi così ottenuti e i tempi/eventi reali.

Questa trasformazione è necessaria perché il C‑index richiede un ordinamento monotono del rischio, mentre le uscite di `XGBSEDebiasedBCE` sono matrici di probabilità di sopravvivenza.

La cross‑validation (`cv=5`) valuta ogni combinazione di parametri con questa metrica, e la migliore viene selezionata come modello finale.

---

### Fase 5: Valutazione Finale e Salvataggio dell’Artefatto

Dopo l’addestramento, il modello viene valutato su training e test calcolando:

- il C‑index sul training (`ci_train`);
- il C‑index sul test (`ci_test`);
- il C‑index medio della cross‑validation (`best_cv_ci`).

Per garantire la piena riproducibilità delle analisi successive (in particolare **SHAP** e **explainability**), viene salvato un **artefatto completo** in formato pickle contenente:

- il modello ottimizzato (`best_model`);
- l’elenco delle feature utilizzate (`feature_cols`);
- i dataframe di training e test (`X_train_df`, `X_test_df`);
- gli oggetti strutturati di sopravvivenza (`y_train`, `y_test`) e le loro versioni scomposte in tempi ed eventi (`t_train`, `e_train`, `t_test`, `e_test`);
- i punteggi di performance;
- i migliori parametri e i semi casuali.

Questo approccio permette, nella fase di valutazione, di ricostruire esattamente il contesto di addestramento e di fornire a SHAP il dataset di background corretto (tipicamente il training set) senza dover ricaricare e riprocessare i dati.

In aggiunta, viene esportata una copia del solo modello con `joblib` per un eventuale deploy leggero.

---

**Riepilogo delle scelte principali**:

- **Nessuna selezione manuale delle feature** – si sfrutta la gestione nativa di XGBoost per i NaN e la sua capacità di ignorare feature irrilevanti.
- **Campionamento deterministico degli iperparametri** – lista pre‑generata di 150 combinazioni per garantire riproducibilità e copertura controllata.
- **Parametri orientati alla regolarizzazione** – valori bassi di `learning_rate`, alberi poco profondi, `min_child_weight` elevato, penalità L1/L2.
- **Funzione di scoring ad hoc** – trasforma le curve di sopravvivenza in rischio per il calcolo del C‑index.
- **Artefatto completo** – include dati, parametri e metadati per explainability e audit.

L’esecuzione di questa pipeline produce un modello XGBSE ottimizzato e tutto il necessario per le successive fasi di validazione clinica e interpretazione.

In [3]:
import os, pickle, warnings, random
import numpy as np
import pandas as pd
import joblib
from sksurv.util import Surv
from sksurv.metrics import concordance_index_censored
from xgbse import XGBSEDebiasedBCE
from sklearn.model_selection import train_test_split, RandomizedSearchCV

warnings.filterwarnings("ignore")

# ==============================================================================
# CONFIGURAZIONE E PATHS
# ==============================================================================
DATA_PATH    = "../data/dataset.csv"
MODEL_DIR    = "../models"
MODEL_PATH   = os.path.join(MODEL_DIR, "xgbse_model.pkl")

os.makedirs(MODEL_DIR, exist_ok=True)

TIME_COL     = "follow_ups.days_to_follow_up"
EVENT_COL    = "demographic.vital_status_DEAD"
RANDOM_STATE = 25
TEST_SIZE    = 0.25
N_ITER       = 150

# ==============================================================================
# FASE 1: CARICAMENTO E PREPARAZIONE
# ==============================================================================
print("=" * 62)
print("  TRAINING PIPELINE - XGBSE (DebiasedBCE)")
print("=" * 62)

df = pd.read_csv(DATA_PATH).replace(-1, np.nan)

excluded_cols  = [TIME_COL, EVENT_COL, "cases.case_id"]
FEATURE_COLS   = [c for c in df.select_dtypes(include=[np.number]).columns if c not in excluded_cols]

X_all = df[FEATURE_COLS].copy()
y_all = Surv.from_arrays(event=df[EVENT_COL].astype(bool), time=df[TIME_COL])

print(f"  [+] Feature utilizzate: {len(FEATURE_COLS)} (nessuna feature selection — XGBoost gestisce NaN nativamente)")

# ==============================================================================
# FASE 2: SPLIT TRAIN / TEST
# ==============================================================================
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df[EVENT_COL],
)

# ==============================================================================
# FASE 3: GRIGLIA IPERPARAMETRI E OTTIMIZZAZIONE
# ==============================================================================
# Risk score scalare: complemento a 1 della sopravvivenza media lungo la griglia temporale.
def risk_from_surv(preds_df):
    return 1.0 - preds_df.values.mean(axis=1)

# Scorer per la ricerca: C-index calcolato sul risk score derivato dalle curve predette.
def score_xgbse(estimator, X_val, y_val):
    p_df     = estimator.predict(X_val)
    r_scores = risk_from_surv(p_df)
    return concordance_index_censored(y_val["event"], y_val["time"], r_scores)[0]

# Lista di N_ITER configurazioni generate con seme fisso, per una ricerca riproducibile.
rng = random.Random(RANDOM_STATE)
xgb_params_list = []

for _ in range(N_ITER):
    params = {
        "objective":        "survival:cox",
        "eval_metric":      "cox-nloglik",
        "tree_method":      "exact",
        "n_estimators":     rng.choice([200, 300, 400, 500]),
        "learning_rate":    rng.choice([0.005, 0.01, 0.02, 0.03]),
        "max_depth":        rng.choice([2, 3, 4]),
        "min_child_weight": rng.choice([5, 10, 15, 20, 30]),
        "gamma":            rng.choice([0.5, 1, 2, 5]),
        "subsample":        rng.choice([0.5, 0.6, 0.7, 0.8]),
        "colsample_bynode": rng.choice([0.3, 0.5, 0.7, 1.0]),
        "reg_alpha":        rng.choice([0.5, 1, 5, 10, 20]),
        "reg_lambda":       rng.choice([1, 5, 10, 20, 50]),
        "seed":             RANDOM_STATE,
    }
    xgb_params_list.append(params)

param_distributions = {"xgb_params": xgb_params_list}

search = RandomizedSearchCV(
    XGBSEDebiasedBCE(),
    param_distributions,
    n_iter=N_ITER,
    cv=5,
    scoring=score_xgbse,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=0,
)
search.fit(X_train, y_train)

best_model  = search.best_estimator_
best_params = search.best_params_["xgb_params"]
best_cv_ci  = search.best_score_

# ==============================================================================
# FASE 4: VALUTAZIONE FINALE E SALVATAGGIO
# ==============================================================================
preds_test  = best_model.predict(X_test)
risk_test   = risk_from_surv(preds_test)
ci_test     = concordance_index_censored(y_test["event"], y_test["time"], risk_test)[0]

preds_train = best_model.predict(X_train)
risk_train  = risk_from_surv(preds_train)
ci_train    = concordance_index_censored(y_train["event"], y_train["time"], risk_train)[0]

# Tempi ed eventi come array separati, salvati nell'artefatto per la fase di valutazione.
t_train = np.array([s[1] for s in y_train])
e_train = np.array([s[0] for s in y_train]).astype(int)
t_test  = np.array([s[1] for s in y_test])
e_test  = np.array([s[0] for s in y_test]).astype(int)

artifact = {
    "model":        best_model,
    "feature_cols": FEATURE_COLS,
    "X_train_df":   X_train,
    "X_test_df":    X_test,
    "y_train":      y_train,
    "y_test":       y_test,
    "t_train":      t_train,
    "e_train":      e_train,
    "t_test":       t_test,
    "e_test":       e_test,
    "cv_c_index":   best_cv_ci,
    "ci_train":     ci_train,
    "ci_test":      ci_test,
    "best_params":  best_params,
    "random_state": RANDOM_STATE,
    "test_size":    TEST_SIZE,
}

with open(MODEL_PATH, "wb") as f:
    pickle.dump(artifact, f)

joblib.dump(best_model, os.path.join(MODEL_DIR, "xgbse_model_joblib.pkl"))

print("=" * 62)
print("  RISULTATI XGBSE (DebiasedBCE)")
print("=" * 62)
print(f"  Feature totali    : {len(FEATURE_COLS)}")
print(f"  CV C-index (best) : {best_cv_ci:.4f}")
print(f"  C-index TRAIN     : {ci_train:.4f}")
print(f"  C-index TEST      : {ci_test:.4f}")
print(f"  Artefatti salvati in: {MODEL_DIR}/")
print("=" * 62)

  TRAINING PIPELINE - XGBSE (DebiasedBCE)
  [+] Feature utilizzate: 33 (nessuna feature selection — XGBoost gestisce NaN nativamente)
  RISULTATI XGBSE (DebiasedBCE)
  Feature totali    : 33
  CV C-index (best) : 0.6584
  C-index TRAIN     : 0.9130
  C-index TEST      : 0.7070
  Artefatti salvati in: ../models/


# XGBoost Survival Embeddings (XGBSE) – Evaluation & Explainable AI

Questa seconda parte del notebook valuta le performance e analizza le logiche decisionali del modello XGBSE addestrato nella cella precedente.
L’analisi comprende C-index, Integrated Brier Score e AUC tempo-dipendente, oltre all’explainability (Permutation Importance e SHAP), e sfrutta l’artefatto di training, che contiene **tutte le feature originali** e i dati necessari per la ricostruzione del contesto.

---

### Fase 1: Caricamento dell’Artefatto e Ricostruzione del Test Set

L’artefatto salvato al termine del training viene ricaricato. Esso contiene:

- il modello ottimizzato (`model`);
- l’elenco completo delle feature (`feature_cols`), corrispondenti a **tutte le variabili numeriche** del dataset (nessuna selezione manuale);
- i dataframe di training e test (`X_train_df`, `X_test_df`);
- tempi ed eventi separati (`t_train`, `e_train`, `t_test`, `e_test`) e gli oggetti strutturati per `scikit-survival`.

Poiché XGBoost gestisce nativamente i valori mancanti e non richiede scaling, non sono necessari ulteriori preprocessori.

Dal modello XGBSE otteniamo, per ogni paziente del test set, un dataframe `preds_test` in cui ogni colonna rappresenta un **istante temporale** e il valore è la probabilità di sopravvivenza stimata $S(t)$.
Per ottenere un **punteggio di rischio** scalare (necessario per il ranking e per molte metriche) si utilizza la trasformazione:

$$
\text{risk} = 1 - \frac{1}{T}\sum_{t} S(t)
$$

ovvero il complemento a 1 della media delle probabilità di sopravvivenza lungo tutti i tempi discreti. Questo approccio sintetizza l’intera curva in un singolo valore monotono rispetto al rischio atteso.

---

### Fase 2: Metriche Globali di Performance

Vengono calcolate due metriche fondamentali per la valutazione di modelli di sopravvivenza:

- **C-index** (Concordance Index): misura la capacità del modello di ordinare correttamente i pazienti in base al rischio. Viene calcolato direttamente sui `risk_scores` derivati dalle curve predette.
- **Integrated Brier Score (IBS)**: valuta l’accuratezza delle probabilità di sopravvivenza lungo l’intero arco temporale. Poiché le predizioni di XGBSE sono disponibili su una griglia di tempi discreta, si interpola linearmente su una griglia comune di 20 punti compresi tra il 10° e il 90° percentile dei tempi di test. I valori vengono poi ritagliati (`clip`) per evitare instabilità numeriche. L’IBS è particolarmente indicato per il modello `XGBSEDebiasedBCE`, che è specificamente progettato per produrre curve ben calibrate.

---

### Fase 2b: AUC Tempo‑dipendente a Orizzonti Clinici

Per apprezzare la capacità discriminativa del modello a orizzonti temporalmente rilevanti (1, 3 e 5 anni), si utilizza la funzione `cumulative_dynamic_auc` di `scikit-survival`.
Vengono considerati solo gli orizzonti che cadono entro il follow‑up massimo disponibile (sia nel training che nel test). Per ogni orizzonte valido si riporta l’AUC e la loro media.
Questa metrica integra il C‑index fornendo una visione più granulare della performance a finestre temporali di interesse clinico.

---

### Fase 3: Stratificazione del Rischio e Visualizzazione delle Curve di Sopravvivenza

I pazienti del test set vengono suddivisi in due gruppi in base alla **mediana** del risk score:

- **Alto Rischio**: risk score > mediana
- **Basso Rischio**: risk score ≤ mediana

Per ciascun gruppo si calcola la curva di sopravvivenza media semplicemente aggregando (mediando) le singole curve predette.
Il grafico a gradini risultante mostra chiaramente la separazione tra le due coorti: un modello ben calibrato deve produrre curve distinte e ordinabili, validando visivamente la capacità di stratificazione clinica.

---

### Fase 4: Explainability Globale – Permutation Feature Importance

Per identificare quali variabili guidano maggiormente le predizioni a livello di popolazione, si applica la **Permutation Feature Importance**.
Poiché il metodo richiede un estimatore con interfaccia `fit`/`predict`, viene creato un wrapper leggero che:

- ignora il `fit` (il modello è già addestrato);
- nella `predict` restituisce i risk scores a partire dalle curve di sopravvivenza.

La funzione di scoring utilizzata è il C‑index. Per ogni feature, i suoi valori vengono permutati 10 volte e la riduzione media del C‑index (rispetto alla baseline) viene convertita in **importanza percentuale relativa**.
Il grafico a barre orizzontali (con deviazione standard) evidenzia i principali driver predittivi, confermando o arricchendo le conoscenze cliniche.

---

### Fase 5: Explainability Locale – SHAP (SHapley Additive exPlanations)

Per comprendere come ciascuna feature contribuisce al rischio di un **singolo paziente**, si utilizza SHAP.
Viene estratto un **background dataset** di 100 campioni dal training set (campionamento casuale con seed fisso) per inizializzare l’`Explainer`. L’explainer è costruito sulla funzione che trasforma le feature in risk score (stessa logica del wrapper precedente).

- **SHAP Summary Plot (beeswarm)**: mostra, per ogni feature, la distribuzione dei valori SHAP sull’intero test set. Il colore indica il valore della feature (rosso = alto, blu = basso), rivelando l’effetto monotono e l’importanza globale.
- **SHAP Waterfall Plot**: generato per i due pazienti con il rischio **più alto** e **più basso** nel test set. Questi grafici partono dal valore base (il rischio medio atteso sulla popolazione di background) e aggiungono/sottraggono i contributi SHAP di ogni feature, illustrando in modo trasparente il ragionamento matematico che porta al rischio finale del singolo paziente.

Tutti i grafici vengono salvati nella cartella `results/` per una successiva inclusione in report o presentazioni.

---

**Riepilogo dei punti principali**:

- **IBS** e **AUC tempo-dipendente** per valutare calibrazione e discriminazione nel tempo.
- Utilizzo di **tutte le feature** (nessuna selezione a monte), coerentemente con la pipeline di training.
- **Permutation Importance** con visualizzazione dell’importanza percentuale.
- **SHAP** con background sampling e generazione automatica di waterfall per i pazienti estremi.
- Salvataggio strutturato di tutti gli output nella directory `results/`.

L’esecuzione di questo notebook fornisce un quadro completo delle performance e delle logiche del modello XGBSE, soddisfacendo sia le esigenze di validazione quantitativa che quelle di interpretabilità clinica.

In [1]:
import os, pickle, warnings
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt
from sklearn.inspection import permutation_importance
from sksurv.util import Surv
from sksurv.metrics import (concordance_index_censored,
                            integrated_brier_score,
                            cumulative_dynamic_auc)

warnings.filterwarnings("ignore")

# ==============================================================================
# CONFIGURAZIONE E PATHS
# ==============================================================================
RESULTS_DIR = "results"
MODEL_DIR = "../models"
MODEL_PATH = os.path.join(MODEL_DIR, "xgbse_model.pkl")

os.makedirs(RESULTS_DIR, exist_ok=True)

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Artefatto non trovato: {MODEL_PATH}. Esegui prima il training.")

TIME_COL = "follow_ups.days_to_follow_up"
EVENT_COL = "demographic.vital_status_DEAD"

# ==============================================================================
# FASE 1: CARICAMENTO ARTEFATTO
# ==============================================================================
print("=" * 62)
print("  EVALUATION & XAI - XGBSE (DebiasedBCE)")
print("=" * 62)

with open(MODEL_PATH, "rb") as f:
    art = pickle.load(f)

model = art["model"]
feature_cols = art["feature_cols"]
X_train_df = art["X_train_df"]
X_test = art["X_test_df"]
y_train = art["y_train"]
t_train = art["t_train"]
e_train = art["e_train"]
t_test = art["t_test"]
e_test = art["e_test"]
RANDOM_STATE = art["random_state"]


# Risk score scalare: complemento a 1 della sopravvivenza media lungo la griglia temporale.
def risk_from_surv(preds_df):
    return 1.0 - preds_df.values.mean(axis=1)


y_train_sksurv = Surv.from_arrays(event=e_train.astype(bool), time=t_train)
y_test_sksurv = Surv.from_arrays(event=e_test.astype(bool), time=t_test)

preds_test = model.predict(X_test)
risk_scores = risk_from_surv(preds_test)

print(f"  [+] Artefatto caricato | CV C-index: {art['cv_c_index']:.4f} | Feature: {len(feature_cols)}")
print(f"  [+] Test set: {len(X_test)} pz | {e_test.sum()} eventi\n")

# ==============================================================================
# FASE 2: METRICHE GLOBALI
# ==============================================================================
print("=" * 62)
print("  FASE 2: METRICHE GLOBALI")
print("=" * 62)

c_index = concordance_index_censored(e_test.astype(bool), t_test, risk_scores)[0]

lb = np.percentile(t_test, 10)
ub = np.percentile(t_test, 90)
ibs_times = np.linspace(lb, ub, 20)

# Curve interpolate sulla griglia dell'IBS e ritagliate per evitare instabilità numeriche.
preds_interp = np.clip(
    np.array([
        np.interp(ibs_times, preds_test.columns.astype(float), preds_test.iloc[i].values)
        for i in range(len(preds_test))
    ]),
    1e-6, 1 - 1e-6,
)

ibs = integrated_brier_score(y_train_sksurv, y_test_sksurv, preds_interp, ibs_times)

print(f"  C-index (test set) : {c_index:.4f}")
print(f"  IBS                : {ibs:.4f} (baseline ~0.25, ↓ meglio)")

# ==============================================================================
# FASE 2b: AUC TIME-DEPENDENT (1, 3, 5 ANNI)
# ==============================================================================
print("\n" + "=" * 62)
print("  FASE 2b: AUC TIME-DEPENDENT")
print("=" * 62)

# Si valutano solo gli orizzonti compresi nel follow-up osservato sia in training sia in test.
time_max_safe = min(t_train.max(), t_test.max())
horizon_map = {1: 365, 3: 1095, 5: 1825}
valid_h = {lbl: d for lbl, d in horizon_map.items() if t_test.min() < d < time_max_safe}
skipped = [lbl for lbl, d in horizon_map.items() if d >= time_max_safe]

if skipped:
    print(f"  [!] Orizzonti esclusi (oltre il follow-up massimo): {skipped} anni")

if valid_h:
    auc_vals, mean_auc = cumulative_dynamic_auc(
        y_train_sksurv, y_test_sksurv, risk_scores,
        np.array(list(valid_h.values()), dtype=float)
    )
    for lbl, auc in zip(valid_h.keys(), auc_vals):
        print(f"  AUC a {lbl} anno/i ({valid_h[lbl]} giorni) : {auc:.4f}")
    print(f"  AUC media (orizzonti validi)           : {mean_auc:.4f}")
else:
    print("  [!] Nessun orizzonte temporale valido.")

# ==============================================================================
# FASE 3: STRATIFICAZIONE DEL RISCHIO
# ==============================================================================
print("\n" + "=" * 62)
print("  FASE 3: STRATIFICAZIONE CLINICA")
print("=" * 62)

# Due gruppi di rischio, separati dalla mediana del risk score sul test set.
median_risk = np.median(risk_scores)
mask_high = risk_scores > median_risk
mask_low = ~mask_high
times_plot = preds_test.columns.values.astype(float)
mean_surv_high = preds_test[mask_high].mean(axis=0)
mean_surv_low = preds_test[mask_low].mean(axis=0)

fig1, ax1 = plt.subplots(figsize=(10, 6))
ax1.step(times_plot, mean_surv_low, where="post", label="Basso Rischio", color="#2ca02c", lw=2.5)
ax1.step(times_plot, mean_surv_high, where="post", label="Alto Rischio", color="#d62728", lw=2.5)
ax1.fill_between(times_plot, mean_surv_low, mean_surv_high, alpha=0.08, color="grey", step="post")
ax1.set_title("Curve di Sopravvivenza (XGBSE): Alto vs Basso Rischio", fontsize=13, fontweight="bold")
ax1.set_ylabel("Probabilità di Sopravvivenza", fontsize=11)
ax1.set_xlabel("Giorni di Follow-up", fontsize=11)
ax1.set_ylim(0, 1.02)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.35, linestyle="--")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "xgbse_survival_curves.png"), dpi=150)
plt.close()
print(f"  [+] Grafico curve salvato in: {RESULTS_DIR}")

# ==============================================================================
# FASE 4: XAI GLOBALE (PERMUTATION FEATURE IMPORTANCE)
# ==============================================================================
print("\n" + "=" * 62)
print("  FASE 4: PERMUTATION FEATURE IMPORTANCE")
print("=" * 62)
print("  Calcolo in corso (10 ripetizioni)...")


# Wrapper con interfaccia scikit-learn: fit non fa nulla, predict restituisce il risk score.
class _XGBSEWrapper:
    def fit(self, X, y=None): return self

    def predict(self, X):
        X_df = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X, columns=feature_cols)
        return risk_from_surv(model.predict(X_df))


# Scorer per la permutation importance: C-index sul risk score.
def _score_fn(estimator, X, y):
    return concordance_index_censored(y["event"], y["time"], estimator.predict(X))[0]


perm = permutation_importance(
    _XGBSEWrapper(), X_test, y_test_sksurv,
    scoring=_score_fn,
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

imp_mean = np.maximum(perm.importances_mean, 0)
imp_pct = 100.0 * imp_mean / (imp_mean.sum() + 1e-12)

df_imp = pd.DataFrame({
    "Feature": feature_cols,
    "Importanza (%)": imp_pct,
    "Std": perm.importances_std,
}).sort_values("Importanza (%)", ascending=False)

df_imp_top = df_imp.head(15)

fig2, ax2 = plt.subplots(figsize=(9, 6))
ax2.barh(df_imp_top["Feature"][::-1], df_imp_top["Importanza (%)"][::-1],
         color="#1f77b4", xerr=df_imp_top["Std"][::-1], capsize=3)
ax2.set_xlabel("Importanza relativa (%)", fontsize=11)
ax2.set_title("Permutation Feature Importance – XGBSE", fontsize=12, fontweight="bold")
ax2.grid(axis="x", alpha=0.35, linestyle="--")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "xgbse_permutation_importance.png"), dpi=150)
plt.close()
print(f"  [+] Grafico Permutation salvato in: {RESULTS_DIR}")

# ==============================================================================
# FASE 5: XAI LOCALE (SHAP)
# ==============================================================================
print("\n" + "=" * 62)
print("  FASE 5: XAI LOCALE (SHAP)")
print("=" * 62)


# Funzione di predizione per SHAP: accetta DataFrame o array e restituisce il risk score.
def _predict_risk_shap(X_raw):
    X_df = X_raw if isinstance(X_raw, pd.DataFrame) else pd.DataFrame(X_raw, columns=feature_cols)
    return risk_from_surv(model.predict(X_df))


background = shap.sample(X_train_df, 100, random_state=RANDOM_STATE)
explainer_shap = shap.Explainer(_predict_risk_shap, background)

print("  Calcolo SHAP su tutto il test set...")
shap_values_all = explainer_shap(X_test)

plt.figure(figsize=(10, 6))
shap.plots.beeswarm(shap_values_all, max_display=15, show=False)
plt.title("SHAP Summary – XGBSE (Contributo al Risk Score)", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "xgbse_shap_summary.png"), dpi=150, bbox_inches="tight")
plt.close()
print("  [+] SHAP Summary plot salvato")

idx_alto = int(np.argmax(risk_scores))
idx_basso = int(np.argmin(risk_scores))

for label, idx in [("ALTO RISCHIO", idx_alto), ("BASSO RISCHIO", idx_basso)]:
    plt.figure(figsize=(14, 7))
    try:
        shap.plots.waterfall(shap_values_all[idx], max_display=13, show=False, matplotlib=True)
    except TypeError:
        shap.plots.waterfall(shap_values_all[idx], max_display=13, show=False)

    ax_wf = plt.gca()
    plt.title(f"SHAP Waterfall – XGBSE – Paziente {label}", fontsize=12, fontweight="bold", pad=12)
    ax_wf.tick_params(axis="y", labelsize=9)
    for text in ax_wf.texts:
        if "f(x)" not in text.get_text() and "base value" not in text.get_text():
            text.set_fontsize(8)
    plt.subplots_adjust(left=0.42, right=0.95, top=0.90, bottom=0.10)
    ax_wf.grid(axis="x", alpha=0.3, linestyle="--")

    fname = f"xgbse_shap_waterfall_{label.lower().replace(' ', '_')}.png"
    plt.savefig(os.path.join(RESULTS_DIR, fname), dpi=150, bbox_inches="tight")
    plt.close()

print("  [+] SHAP Waterfall plots salvati per pazienti ad alto e basso rischio")

# ==============================================================================
# RIEPILOGO FINALE
# ==============================================================================
print("\n" + "=" * 62)
print("  RIEPILOGO FINALE – XGBSE (DebiasedBCE)")
print("=" * 62)
print(f"  Esecuzione completata. Tutti i grafici sono salvati in:\n  {RESULTS_DIR}")

  EVALUATION & XAI - XGBSE (DebiasedBCE)
  [+] Artefatto caricato | CV C-index: 0.6584 | Feature: 33
  [+] Test set: 132 pz | 56 eventi

  FASE 2: METRICHE GLOBALI
  C-index (test set) : 0.7070
  IBS                : 0.1806 (baseline ~0.25, ↓ meglio)

  FASE 2b: AUC TIME-DEPENDENT
  AUC a 1 anno/i (365 giorni) : 0.7128
  AUC a 3 anno/i (1095 giorni) : 0.7592
  AUC a 5 anno/i (1825 giorni) : 0.7593
  AUC media (orizzonti validi)           : 0.7441

  FASE 3: STRATIFICAZIONE CLINICA
  [+] Grafico curve salvato in: results

  FASE 4: PERMUTATION FEATURE IMPORTANCE
  Calcolo in corso (10 ripetizioni)...
  [+] Grafico Permutation salvato in: results

  FASE 5: XAI LOCALE (SHAP)
  Calcolo SHAP su tutto il test set...


PermutationExplainer explainer: 133it [00:13,  3.67it/s]                         


  [+] SHAP Summary plot salvato
  [+] SHAP Waterfall plots salvati per pazienti ad alto e basso rischio

  RIEPILOGO FINALE – XGBSE (DebiasedBCE)
  Esecuzione completata. Tutti i grafici sono salvati in:
  results
